# Day 1 — Intake & Canonicalization: Trust Nothing, Parse Everything

**Course:** Agentic Systems for Forward Deployed Engineers  
**Theme:** Before any automation can act on an invoice, someone — or something — must decide it is trustworthy. Day 1 defines that boundary.

---

## What this day covers

| Layer | Skill |
|---|---|
| Data modelling | Pydantic `BaseModel` with strict validation, frozen models, custom validators |
| LLM integration | Structured extraction via Azure OpenAI (string-heavy on purpose) |
| Deterministic normalization | Money parsing (locale-safe), date parsing, currency symbol mapping, evidence grounding |
| Intake boundary | A `CanonicalInvoice` is the only object that crosses to Day 2 — rejection is the safe default |

## Why this matters for FDEs

Agentic systems fail silently when untrusted data crosses a service boundary unchecked.  
The Day 1 pattern — **LLM extracts, deterministic Python normalises, Pydantic validates** — is a template you will reuse on every system you build.

---

## Prerequisites

```bash
pip install -e .[dev]   # or: pip install pydantic
```
No Azure credentials are needed for the exercises below — we work with fixtures and the deterministic layer only.

In [1]:
# ── environment bootstrap ──────────────────────────────────────────────────────
import sys
from pathlib import Path

# Make the src/ tree importable regardless of where this notebook is opened
repo_root = Path().resolve().parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

print("✓ repo root:", repo_root)
print("✓ Python:", sys.version.split()[0])

✓ repo root: /workspaces/agentic-accounts-payable-orchestrator
✓ Python: 3.12.13


---
## Part 1 — The Data Models

### Why string-heavy extraction?

When an LLM sees `1.250,00` it might return `1250.0` or `1.25` depending on training data locale.  
We give the model a schema that is **deliberately string-heavy** (`net_amount_text: str`) so that *deterministic Python* owns parsing.  
The model's only job is to **copy** text from the source — not interpret it.

The pipeline is three steps:

```
InvoicePackageInput   →   ExtractedInvoiceCandidate   →   CanonicalInvoice
  (raw email/PDF)          (LLM string copy)              (typed, validated)
```

In [2]:
from aegisap.day_01.models import (
    AttachmentInput,
    InvoicePackageInput,
    ExtractedInvoiceCandidate,
    EvidenceValue,
    CanonicalInvoice,
)
import json

# Inspect what each model expects
for model in [InvoicePackageInput, ExtractedInvoiceCandidate, CanonicalInvoice]:
    fields = list(model.model_fields.keys())
    print(f"\n{model.__name__}")
    print("  fields:", fields)


InvoicePackageInput
  fields: ['message_id', 'email_subject', 'email_body', 'attachments']

ExtractedInvoiceCandidate
  fields: ['supplier_name_text', 'invoice_number_text', 'invoice_date_text', 'currency_text', 'net_amount', 'vat_amount', 'gross_amount', 'po_reference_text', 'bank_details_text']

CanonicalInvoice
  fields: ['supplier_name', 'invoice_number', 'invoice_date', 'currency', 'net_amount', 'vat_amount', 'gross_amount', 'po_reference', 'bank_details_hash']


In [3]:
# Build a minimal invoice package to understand the shape
package = InvoicePackageInput(
    message_id="MSG-001",
    email_subject="Invoice INV-2024-001 from Contoso Ltd",
    email_body="Please find attached invoice INV-2024-001 for £1,234.56 due 2024-03-01.",
    attachments=[
        AttachmentInput(
            filename="invoice.pdf",
            content_type="application/pdf",
            sha256="a" * 64,  # stub hash
            extracted_text=(
                "INVOICE\n"
                "Supplier: Contoso Ltd\n"
                "Invoice No: INV-2024-001\n"
                "Date: 2024-03-01\n"
                "Net: £1,050.00\n"
                "VAT: £184.56\n"
                "Gross: £1,234.56\n"
                "PO: PO-9901\n"
                "Bank: GB29NWBK60161331926819"
            ),
        )
    ],
)

print("Package created ✓")
print(json.dumps(package.model_dump(mode="json"), indent=2))

Package created ✓
{
  "message_id": "MSG-001",
  "email_subject": "Invoice INV-2024-001 from Contoso Ltd",
  "email_body": "Please find attached invoice INV-2024-001 for \u00a31,234.56 due 2024-03-01.",
  "attachments": [
    {
      "filename": "invoice.pdf",
      "content_type": "application/pdf",
      "sha256": "aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa",
      "extracted_text": "INVOICE\nSupplier: Contoso Ltd\nInvoice No: INV-2024-001\nDate: 2024-03-01\nNet: \u00a31,050.00\nVAT: \u00a3184.56\nGross: \u00a31,234.56\nPO: PO-9901\nBank: GB29NWBK60161331926819"
    }
  ]
}


---
## Part 2 — Deterministic Normalization

This is the most security-critical layer.  
The normalizers enforce two invariants:
1. **Evidence grounding** — every extracted value must be *findable* in the original source text.
2. **Locale-safe money parsing** — ambiguous formats (e.g. `1.250`) are **rejected**, not guessed.

### 2a — Money Parsing

In [4]:
from aegisap.day_01.normalizers import parse_money
from decimal import Decimal

test_cases = [
    # (raw_string, expect_success, description)
    ("1,234.56",  True,  "US format — unambiguous"),
    ("1.234,56",  True,  "European format — unambiguous"),
    ("£1,234.56", True,  "GBP prefix — currency stripped"),
    ("1234.00",   True,  "Plain decimal"),
    ("1.250",     False, "AMBIGUOUS — could be 1.25 or 1250"),
    ("1,250",     False, "AMBIGUOUS — could be 1.25 or 1250"),
    ("-500.00",   False, "Negative — rejected at Day 1"),
]

print(f"{'Raw':<15} {'Expected':<10} {'Result':<12} {'Note'}")
print("-" * 70)
for raw, should_succeed, note in test_cases:
    try:
        result = parse_money(raw)
        status = "✓ PASS" if should_succeed else "✗ FAIL (should reject)"
        print(f"{raw:<15} {str(should_succeed):<10} {str(result):<12} {note}")
    except ValueError as e:
        status = "✓ REJECTED" if not should_succeed else f"✗ FAIL: {e}"
        print(f"{raw:<15} {str(should_succeed):<10} {status:<12} {note}")

Raw             Expected   Result       Note
----------------------------------------------------------------------
1,234.56        True       1234.56      US format — unambiguous
1.234,56        True       1234.56      European format — unambiguous
£1,234.56       True       1234.56      GBP prefix — currency stripped
1234.00         True       1234.00      Plain decimal
1.250           False      ✓ REJECTED   AMBIGUOUS — could be 1.25 or 1250
1,250           False      ✓ REJECTED   AMBIGUOUS — could be 1.25 or 1250
-500.00         False      ✓ REJECTED   Negative — rejected at Day 1


> **Key insight:** `1.250` is ambiguous — it could be `1.25` (2 decimal places) or `1,250.00` (European thousands separator).  
> The Day 1 normalizer **rejects** all such ambiguous values at the intake boundary.  
> This prevents a vendor issuing `1.250` knowing the system interprets it as `1250.00`.

In [5]:
from aegisap.day_01.normalizers import parse_invoice_date, normalize_currency

# Date parsing — tolerates many European and ISO formats
date_cases = [
    "2024-03-01",
    "01/03/2024",
    "01.03.2024",
    "01 Mar 2024",
    "01 March 2024",
    "01-03-2024",
]

print("Date parsing:")
for raw in date_cases:
    print(f"  {raw!r:20} →  {parse_invoice_date(raw)}")

print("\nCurrency normalisation:")
for sym in ["EUR", "GBP", "USD", "€", "£", "$"]:
    print(f"  {sym!r:6} →  {normalize_currency(sym)}")

Date parsing:
  '2024-03-01'         →  2024-03-01
  '01/03/2024'         →  2024-03-01
  '01.03.2024'         →  2024-03-01
  '01 Mar 2024'        →  2024-03-01
  '01 March 2024'      →  2024-03-01
  '01-03-2024'         →  2024-03-01

Currency normalisation:
  'EUR'  →  EUR
  'GBP'  →  GBP
  'USD'  →  USD
  '€'    →  EUR
  '£'    →  GBP
  '$'    →  USD


---
## Part 3 — Evidence Grounding

A normalizer is only as good as its ability to prove the extracted value came from the source.  
The `_require_text` and `_has_structured_boundaries` functions ensure:

- A value like `INV-2024-001` must be **literally present** in the email or PDF text.
- Boundary-matching prevents `INV-2024-001` matching inside `PREFIX-INV-2024-001-SUFFIX`.

Let's watch evidence grounding work in practice.

In [6]:
from aegisap.day_01.normalizers import to_canonical_invoice
from aegisap.day_01.models import ExtractedInvoiceCandidate, EvidenceValue

# Candidate that matches the package we built above
good_candidate = ExtractedInvoiceCandidate(
    supplier_name_text="Contoso Ltd",
    invoice_number_text="INV-2024-001",
    invoice_date_text="2024-03-01",
    currency_text="GBP",
    net_amount=EvidenceValue(pdf_text="£1,050.00"),
    vat_amount=EvidenceValue(pdf_text="£184.56"),
    gross_amount=EvidenceValue(pdf_text="£1,234.56", email_text="£1,234.56"),
    po_reference_text="PO-9901",
    bank_details_text="GB29NWBK60161331926819",
)

try:
    invoice = to_canonical_invoice(good_candidate, package)
    print("✓ Canonicalization succeeded")
    print(f"  supplier_name:  {invoice.supplier_name}")
    print(f"  invoice_number: {invoice.invoice_number}")
    print(f"  currency:       {invoice.currency}")
    print(f"  net_amount:     {invoice.net_amount}")
    print(f"  vat_amount:     {invoice.vat_amount}")
    print(f"  gross_amount:   {invoice.gross_amount}")
    print(f"  bank_details_hash: {invoice.bank_details_hash[:16]}...")
except ValueError as e:
    print(f"✗ Rejected: {e}")

✗ Rejected: currency_text evidence not found in source text: 'GBP'


In [7]:
# What happens when the extracted text is NOT in the source?
tampered_candidate = ExtractedInvoiceCandidate(
    supplier_name_text="Contoso Ltd",
    invoice_number_text="INV-2024-001",
    invoice_date_text="2024-03-01",
    currency_text="GBP",
    net_amount=EvidenceValue(pdf_text="£9,999.99"),  # ← not in source!
    vat_amount=EvidenceValue(pdf_text="£184.56"),
    gross_amount=EvidenceValue(pdf_text="£1,234.56"),
    po_reference_text="PO-9901",
    bank_details_text="GB29NWBK60161331926819",
)

try:
    invoice = to_canonical_invoice(tampered_candidate, package)
    print("DANGER — accepted a tampered value!")
except ValueError as e:
    print(f"✓ Correctly rejected: {e}")

✓ Correctly rejected: currency_text evidence not found in source text: 'GBP'


---
## Part 4 — Loading Real Fixtures

The `fixtures/` directory contains pre-built test cases that represent real-world edge cases.  
Use the fixture loader to explore them interactively.

In [8]:
from aegisap.day_01.fixture_loader import list_fixture_cases, load_fixture_package, load_fixture_candidate
from aegisap.day_01.service import canonicalize_with_candidate, IntakeReject

cases = list_fixture_cases()
print(f"Available fixture cases: {cases}\n")

for case_name in cases:
    pkg = load_fixture_package(case_name)
    cand = load_fixture_candidate(case_name)
    try:
        inv = canonicalize_with_candidate(pkg, cand)
        print(f"  ✓ {case_name:<20} → {inv.supplier_name} | {inv.currency} {inv.gross_amount} | PO: {inv.po_reference}")
    except IntakeReject as e:
        print(f"  ✗ {case_name:<20} → REJECTED")

Available fixture cases: ['happy_path', 'locale_mismatch', 'missing_po']

  ✓ happy_path           → Contoso Ltd | GBP 1500.00 | PO: PO-7781
  ✓ locale_mismatch      → Rhein Energie GmbH | EUR 1250.00 | PO: PO-9001
  ✗ missing_po           → REJECTED


---
## Part 5 — Amount Reconciliation Across Sources

When an amount appears in **both** the PDF and the email, their parsed values must agree.  
If they differ — even by 1 cent — the invoice is rejected.  
This protects against a class of fraud where an email states one amount but the PDF states another.

In [9]:
from aegisap.day_01.normalizers import reconcile_amount
from aegisap.day_01.models import EvidenceValue, InvoicePackageInput, AttachmentInput

# Build a package where PDF and email agree
agree_package = InvoicePackageInput(
    message_id="MSG-002",
    email_subject="Test",
    email_body="Total: 5,000.00 EUR",
    attachments=[
        AttachmentInput(
            filename="inv.pdf",
            content_type="application/pdf",
            sha256="b" * 64,
            extracted_text="Amount: 5,000.00",
        )
    ],
)

print("=== Consistent sources ===")
try:
    amount = reconcile_amount(
        EvidenceValue(pdf_text="5,000.00", email_text="5,000.00"),
        agree_package,
        field_name="gross_amount",
    )
    print(f"✓ Reconciled: {amount}")
except ValueError as e:
    print(f"✗ Failed: {e}")

print("\n=== Mismatched sources ===")
try:
    amount = reconcile_amount(
        EvidenceValue(pdf_text="5,000.00", email_text="5,000.01"),
        agree_package,
        field_name="gross_amount",
    )
    print(f"Accepted: {amount}  ← DANGER")
except ValueError as e:
    print(f"✓ Correctly rejected: {e}")

=== Consistent sources ===
✓ Reconciled: 5000.00

=== Mismatched sources ===
✓ Correctly rejected: gross_amount.email_text evidence not found in source text: '5,000.01'


---
## Part 6 — The Intake Boundary

The `run_day_01_intake` service wraps everything into a single entry point.  
It calls the LLM extraction agent, normalizes, validates, and either returns a `CanonicalInvoice` — or raises `IntakeReject`.

In production you would call `run_day_01_intake(package)`.  
In tests we bypass the LLM with `canonicalize_with_candidate(package, candidate)`.  

This separation — **same normalization path, swappable extraction source** — is the key architectural decision of Day 1.

In [10]:
# Simulate the full intake boundary using a fixture (no LLM required)
from aegisap.day_01.service import canonicalize_with_candidate, IntakeReject
from aegisap.day_01.fixture_loader import load_fixture_package, load_fixture_candidate

case = "happy_path"
pkg = load_fixture_package(case)
cand = load_fixture_candidate(case)

print(f"Running intake for case: {case!r}")
print(f"Package message_id:      {pkg.message_id}")
print(f"Email subject:           {pkg.email_subject}")
print()

try:
    canonical = canonicalize_with_candidate(pkg, cand)
    print("Intake result: ✓ ACCEPTED")
    print()
    print(canonical.model_dump_json(indent=2))
except IntakeReject as e:
    print(f"Intake result: ✗ REJECTED\n{e}")

Running intake for case: 'happy_path'
Package message_id:      msg-happy-001
Email subject:           Invoice INV-1007 from Contoso Ltd

Intake result: ✓ ACCEPTED

{
  "supplier_name": "Contoso Ltd",
  "invoice_number": "INV-1007",
  "invoice_date": "2026-03-14",
  "currency": "GBP",
  "net_amount": "1250.00",
  "vat_amount": "250.00",
  "gross_amount": "1500.00",
  "po_reference": "PO-7781",
  "bank_details_hash": "8c037924cf9e8573d330f29152dd464c31b8d319a78e6da088d619b36c765e91"
}


---
## Exercises

Complete the following exercises to consolidate Day 1 concepts.

### Exercise 1 — Add a new date format

The `DATE_FORMATS` tuple in `normalizers.py` does not include `MM/DD/YYYY` (US format).  
1. Try parsing `"03/15/2024"` with `parse_invoice_date` — observe the failure.
2. In the cell below, write a function `parse_with_us_format` that tries the existing formats first, then adds `"%m/%d/%Y"` as a fallback. 
3. Consider: should US format be a fallback or a primary format? Why is ordering important?

In [ ]:
# Exercise 1 workspace
from aegisap.day_01.normalizers import parse_invoice_date, DATE_FORMATS
from datetime import datetime

us_date = "03/15/2024"

# Step 1 — observe the failure
try:
    print(parse_invoice_date(us_date))
except ValueError as e:
    print(f"Expected failure: {e}")

# Step 2 — your implementation below
# def parse_with_us_format(raw: str):
#     ...

### Exercise 2 — Build a rejection matrix

Create a table that shows which field, when tampered with, causes the canonicalization to reject.  
You should test at least:
- A supplier name not present in the email or PDF
- A gross amount that doesn't equal net + VAT
- A PO reference that exists in the email but uses a different separator in the PDF
- A bank details string shorter than 8 alphanumeric characters

In [ ]:
# Exercise 2 workspace
from aegisap.day_01.normalizers import to_canonical_invoice, hash_bank_details
from aegisap.day_01.models import ExtractedInvoiceCandidate, EvidenceValue, InvoicePackageInput, AttachmentInput

base_package = InvoicePackageInput(
    message_id="EX2",
    email_subject="Invoice INV-EX2 from Vendor Corp",
    email_body="Invoice INV-EX2. Amount: 1,000.00 EUR. PO: PO-EX2. Date: 2024-04-01.",
    attachments=[
        AttachmentInput(
            filename="inv.pdf",
            content_type="application/pdf",
            sha256="c" * 64,
            extracted_text=(
                "Vendor Corp\nInvoice: INV-EX2\nDate: 2024-04-01\n"
                "Net: 840.34 EUR\nVAT: 159.66 EUR\nGross: 1,000.00 EUR\n"
                "PO: PO-EX2\nBank: DE89370400440532013000"
            ),
        )
    ],
)

# Define tampering scenarios here
# tampered_cases = [...]
# for name, candidate in tampered_cases:
#     ...

### Exercise 3 — Understand the `CanonicalInvoice` contract

`CanonicalInvoice` is **frozen** (immutable after creation) and has `strict=True` and `extra="forbid"`.  
Run the cells below and explain *why* each constraint is there from a security perspective.

In [ ]:
from aegisap.day_01.fixture_loader import load_fixture_package, load_fixture_candidate
from aegisap.day_01.service import canonicalize_with_candidate
from pydantic import ValidationError

inv = canonicalize_with_candidate(load_fixture_package("happy_path"), load_fixture_candidate("happy_path"))

# Test 1 — try mutating a frozen field
print("Testing frozen (immutability):")
try:
    inv.supplier_name = "HACKED"
    print("  DANGER — mutation succeeded!")
except Exception as e:
    print(f"  ✓ Mutation blocked: {type(e).__name__}")

# Test 2 — try creating with an extra unknown field
print("\nTesting extra='forbid':")
try:
    from aegisap.day_01.models import CanonicalInvoice
    data = inv.model_dump(mode="json")
    data["rogue_field"] = "injected"
    CanonicalInvoice.model_validate(data)
    print("  DANGER — extra field accepted!")
except ValidationError as e:
    print(f"  ✓ Extra field rejected: {e.errors()[0]['type']}")

---
## Summary — Day 1 Principles

| Principle | Implementation |
|---|---|
| LLM extracts, Python normalises | `ExtractedInvoiceCandidate` stays string-heavy; `to_canonical_invoice` owns parsing |
| Evidence grounding prevents hallucination | `_require_text` checks every extracted field against source text |
| Ambiguous inputs are rejected, not guessed | `parse_money` rejects `1.250` and `1,250` when ambiguous |
| Immutable output type | `CanonicalInvoice` is `frozen=True` — once created it cannot be silently mutated |
| Single exit boundary | `run_day_01_intake` returns `CanonicalInvoice` or raises `IntakeReject` — no partial objects |

## What Day 2 builds on this

Day 2 receives a `CanonicalInvoice` and builds a **stateful LangGraph workflow** on top of it.  
It can trust the invoice is valid because Day 1 already enforced that guarantee.  
The clean separation of concerns — Day 1 validates, Day 2 routes — is the foundation of the whole system.